# 05 - Load the Renewables-Climate Mart into DuckDB

## Overview

This notebook loads the processed CSV tables created in the previous notebooks into a DuckDB database.

The preceding notebooks produced the dimension and fact tables of the Renewables-Climate Mart as CSV files. This notebook turns those files into a relational DuckDB database by explicitly creating the mart schema, defining primary and foreign key relationships, and loading the processed data into the corresponding tables.

The resulting DuckDB database provides a portable analytical data mart that can be queried directly with SQL and used as the backend for demonstration or NLIDB workflows.

## Input Data

This notebook expects the processed CSV files generated by Notebooks 02–04:

- `data/processed/dim_company_h.csv`
- `data/processed/dim_area_h.csv`
- `data/processed/dim_energy_source_h.csv`
- `data/processed/dim_asset.csv`
- `data/processed/dim_date.csv`
- `data/processed/fact_generation_daily.csv`
- `data/processed/fact_weather_daily.csv`

## Output

The notebook creates the DuckDB database:

- `database/renewables_climate_mart.duckdb`

In [1]:
import os

import duckdb

In [2]:
DATA_DIR = "../data"

PROCESSED_DIR = f"{DATA_DIR}/processed"
DATABASE_DIR = "../database"
os.makedirs(DATABASE_DIR, exist_ok=True)

DB_PATH = f"{DATABASE_DIR}/renewables_climate_mart.duckdb"

## 1. Create DuckDB Schema with Primary and Foreign Keys

This section creates the relational schema of the Renewables-Climate Mart.

The dimension tables are created first and define primary keys for companies, areas, energy sources, assets, and dates. The asset dimension references the company, area, energy source, and commissioning date dimensions. The fact tables reference the dimensions required for analysis: generation records reference assets and dates, while weather records reference areas and dates.

Existing tables are dropped first so the notebook can be rerun and always produces a clean database instance.

In [3]:
con = duckdb.connect(DB_PATH)

# Drop tables in dependency order.
for table_name in [
    "fact_weather_daily",
    "fact_generation_daily",
    "dim_asset",
    "dim_date",
    "dim_energy_source_h",
    "dim_area_h",
    "dim_company_h",
]:
    con.execute(f"DROP TABLE IF EXISTS {table_name};")

In [4]:
con.execute("""
CREATE TABLE dim_company_h (
    company_id INTEGER PRIMARY KEY,
    company_name VARCHAR NOT NULL,
    parent_company VARCHAR,
    company_level VARCHAR
);
""")

con.execute("""
CREATE TABLE dim_area_h (
    area_id INTEGER PRIMARY KEY,
    zipcode INTEGER,
    city VARCHAR,
    state VARCHAR,
    tso_region VARCHAR,
    latitude DOUBLE,
    longitude DOUBLE
);
""")

con.execute("""
CREATE TABLE dim_energy_source_h (
    energy_source_id INTEGER PRIMARY KEY,
    energy_source_name VARCHAR NOT NULL,
    energy_source_group VARCHAR
);
""")

con.execute("""
CREATE TABLE dim_date (
    date_id INTEGER PRIMARY KEY,
    date DATE NOT NULL,
    day_of_year INTEGER,
    weekday VARCHAR,
    weekday_num INTEGER,
    week_iso VARCHAR,
    month_name VARCHAR,
    month INTEGER,
    quarter VARCHAR,
    year INTEGER,
    year_iso INTEGER,
    season VARCHAR,
    is_weekend BOOLEAN,
    is_holiday BOOLEAN,
    is_workday BOOLEAN
);
""")

con.execute("""
CREATE TABLE dim_asset (
    asset_id INTEGER PRIMARY KEY,
    company_id INTEGER NOT NULL,
    area_id INTEGER NOT NULL,
    energy_source_id INTEGER NOT NULL,
    commissioning_date_id INTEGER NOT NULL,
    asset_name VARCHAR,
    generation_capacity_MW DOUBLE,

    FOREIGN KEY (company_id) REFERENCES dim_company_h(company_id),
    FOREIGN KEY (area_id) REFERENCES dim_area_h(area_id),
    FOREIGN KEY (energy_source_id) REFERENCES dim_energy_source_h(energy_source_id),
    FOREIGN KEY (commissioning_date_id) REFERENCES dim_date(date_id)
);
""")

con.execute("""
CREATE TABLE fact_generation_daily (
    date_id INTEGER NOT NULL,
    asset_id INTEGER NOT NULL,
    generation_MWh DOUBLE NOT NULL,

    FOREIGN KEY (date_id) REFERENCES dim_date(date_id),
    FOREIGN KEY (asset_id) REFERENCES dim_asset(asset_id)
);
""")

con.execute("""
CREATE TABLE fact_weather_daily (
    area_id INTEGER NOT NULL,
    date_id INTEGER NOT NULL,
    avg_temperature_C DOUBLE,
    avg_wind_power_density_W_per_m2 DOUBLE,
    total_solar_irradiation_kWh_per_m2 DOUBLE,

    FOREIGN KEY (area_id) REFERENCES dim_area_h(area_id),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id)
);
""")

## 2. Load Data into the Defined Schema

This section loads the processed dimension and fact tables into the previously defined DuckDB schema.

The CSV files generated by the earlier pipeline steps are imported directly into the corresponding tables using DuckDB's `COPY` command. The loading order follows the dependency structure of the mart, ensuring that all referenced dimension records exist before dependent tables are populated.

In [5]:
csv_files = {
    "dim_company_h": "dim_company_h.csv",
    "dim_area_h": "dim_area_h.csv",
    "dim_energy_source_h": "dim_energy_source_h.csv",
    "dim_date": "dim_date.csv",
    "dim_asset": "dim_asset.csv",
    "fact_generation_daily": "fact_generation_daily.csv",
    "fact_weather_daily": "fact_weather_daily.csv",
}

for table_name, file_name in csv_files.items():
    csv_path = f"{PROCESSED_DIR}/{file_name}"

    con.execute(f"""
        COPY {table_name}
        FROM '{csv_path}'
        (HEADER, DELIMITER ',');
    """)

    print(f"Loaded {table_name}")

Loaded dim_company_h
Loaded dim_area_h
Loaded dim_energy_source_h


Loaded dim_date
Loaded dim_asset


Loaded fact_generation_daily


Loaded fact_weather_daily


## 3. Validate Loaded Mart

This section performs basic validation checks after loading the data into DuckDB.

The checks summarize row counts, inspect the defined primary and foreign key constraints, and verify that the loaded fact tables follow their intended grain and coverage assumptions.

In [6]:
row_counts = con.execute("""
SELECT 'dim_company_h' AS table_name, COUNT(*) AS row_count FROM dim_company_h
UNION ALL SELECT 'dim_area_h', COUNT(*) FROM dim_area_h
UNION ALL SELECT 'dim_energy_source_h', COUNT(*) FROM dim_energy_source_h
UNION ALL SELECT 'dim_date', COUNT(*) FROM dim_date
UNION ALL SELECT 'dim_asset', COUNT(*) FROM dim_asset
UNION ALL SELECT 'fact_generation_daily', COUNT(*) FROM fact_generation_daily
UNION ALL SELECT 'fact_weather_daily', COUNT(*) FROM fact_weather_daily
ORDER BY table_name;
""").df()

row_counts

,table_name,row_count
0,dim_area_h,241
1,dim_asset,625
2,dim_company_h,90
3,dim_date,35643
4,dim_energy_source_h,4
5,fact_generation_daily,428693
6,fact_weather_daily,176171


In [7]:
# Show the primary and foreign key definitions stored in DuckDB.
constraints = con.execute("""
SELECT
    table_name,
    constraint_type,
    constraint_column_names,
    referenced_table,
    referenced_column_names
FROM duckdb_constraints()
WHERE table_name IN (
    'dim_company_h',
    'dim_area_h',
    'dim_energy_source_h',
    'dim_date',
    'dim_asset',
    'fact_generation_daily',
    'fact_weather_daily'
)
ORDER BY table_name, constraint_type;
""").df()

constraints

,table_name,constraint_type,constraint_column_names,referenced_table,referenced_column_names
0,dim_area_h,NOT NULL,[area_id],None,[]
1,dim_area_h,PRIMARY KEY,[area_id],None,[]
2,dim_asset,FOREIGN KEY,[company_id],dim_company_h,[company_id]
3,dim_asset,FOREIGN KEY,[area_id],dim_area_h,[area_id]
4,dim_asset,FOREIGN KEY,[energy_source_id],dim_energy_source_h,[energy_source_id]
5,dim_asset,FOREIGN KEY,[commissioning_date_id],dim_date,[date_id]
6,dim_asset,NOT NULL,[company_id],None,[]
7,dim_asset,NOT NULL,[area_id],None,[]
8,dim_asset,NOT NULL,[energy_source_id],None,[]
9,dim_asset,NOT NULL,[commissioning_date_id],None,[]


In [8]:
# Check whether every asset has at most one generation record per day and whether generation exists only for valid asset-date combinations

generation_check = con.execute("""
WITH duplicate_combinations AS (
    SELECT
        asset_id,
        date_id,
        COUNT(*) AS row_count
    FROM fact_generation_daily
    GROUP BY asset_id, date_id
    HAVING COUNT(*) > 1
),

invalid_generation_dates AS (
    SELECT
        fg.asset_id,
        fg.date_id
    FROM fact_generation_daily fg
    JOIN dim_asset a
        ON fg.asset_id = a.asset_id
    WHERE fg.date_id < a.commissioning_date_id
)

SELECT
    'duplicate_asset_date_combinations' AS issue,
    COUNT(*) AS row_count
FROM duplicate_combinations

UNION ALL

SELECT
    'generation_before_commissioning' AS issue,
    COUNT(*) AS row_count
FROM invalid_generation_dates;
""").df()

generation_check

,issue,row_count
0,duplicate_asset_date_combinations,0
1,generation_before_commissioning,0


In [9]:
# Check whether every area has one weather record for every date in the analysis period

coverage_check = con.execute("""
WITH expected AS (
    SELECT
        a.area_id,
        d.date_id
    FROM dim_area_h a
    CROSS JOIN dim_date d
    WHERE d.year BETWEEN 2024 AND 2025
),

missing_combinations AS (
    SELECT
        e.area_id,
        e.date_id
    FROM expected e
    LEFT JOIN fact_weather_daily fw
        ON e.area_id = fw.area_id
       AND e.date_id = fw.date_id
    WHERE fw.area_id IS NULL
),

duplicate_combinations AS (
    SELECT
        area_id,
        date_id,
        COUNT(*) AS row_count
    FROM fact_weather_daily
    GROUP BY area_id, date_id
    HAVING COUNT(*) > 1
)

SELECT
    'missing_area_date_combinations' AS issue,
    COUNT(*) AS row_count
FROM missing_combinations

UNION ALL

SELECT
    'duplicate_area_date_combinations' AS issue,
    COUNT(*) AS row_count
FROM duplicate_combinations;
""").df()

coverage_check

,issue,row_count
0,missing_area_date_combinations,0
1,duplicate_area_date_combinations,0


## 4. Example Queries

This section demonstrates two analytical queries on the completed Renewables-Climate Mart.

The first query summarizes annual generation by energy source and year. The second query combines all tables in the mart and performs a hierarchy-based rollup for 2025, adding regional and weather context.

In [10]:
con.execute("""
SELECT
    d.year,
    es.energy_source_name,
    ROUND(SUM(fg.generation_MWh), 2) AS generation_MWh
FROM fact_generation_daily fg
JOIN dim_date d ON fg.date_id = d.date_id
JOIN dim_asset a ON fg.asset_id = a.asset_id
JOIN dim_energy_source_h es ON a.energy_source_id = es.energy_source_id
GROUP BY d.year, es.energy_source_name
ORDER BY d.year desc, es.energy_source_name;
""").df()

,year,energy_source_name,generation_MWh
0,2025,Biomass,972000.0
1,2025,Hydro,6400.0
2,2025,Solar,54900.0
3,2025,Wind,1271700.0
4,2024,Biomass,647000.0
5,2024,Hydro,6400.0
6,2024,Solar,26300.0
7,2024,Wind,1288300.0


In [11]:
con.execute("""
WITH company_rollup AS (
    -- Holding level: every company rolls up to the holding
    SELECT
        company_id,
        'Holding' AS reporting_level,
        'electricville AG' AS reporting_company
    FROM dim_company_h

    UNION ALL

    -- Direct-subsidiary level: direct subsidiaries keep their own name,
    -- indirect subsidiaries roll up to their parent company
    SELECT
        company_id,
        'Direct subsidiary' AS reporting_level,
        CASE
            WHEN company_level = 'direct Subsidiary' THEN company_name
            WHEN company_level = 'indirect Subsidiary' THEN parent_company
        END AS reporting_company
    FROM dim_company_h
    WHERE company_level IN ('direct Subsidiary', 'indirect Subsidiary')
)

SELECT
    cr.reporting_level,
    cr.reporting_company,
    es.energy_source_name,
    ar.tso_region,
    COUNT(DISTINCT a.asset_id) AS asset_count,
    ROUND(SUM(fg.generation_MWh), 2) AS generation_MWh,
    ROUND(AVG(w.avg_temperature_C), 2) AS avg_temperature_C
FROM fact_generation_daily fg
JOIN dim_asset a
    ON fg.asset_id = a.asset_id
JOIN company_rollup cr
    ON a.company_id = cr.company_id
JOIN dim_date d
    ON fg.date_id = d.date_id
JOIN dim_energy_source_h es
    ON a.energy_source_id = es.energy_source_id
JOIN dim_area_h ar
    ON a.area_id = ar.area_id
JOIN fact_weather_daily w
    ON w.area_id = ar.area_id
   AND w.date_id = d.date_id
WHERE d.year = 2025
GROUP BY
    cr.reporting_level,
    cr.reporting_company,
    es.energy_source_name,
    ar.tso_region
ORDER BY
    cr.reporting_level desc,
    cr.reporting_company desc,
    es.energy_source_name,
    ar.tso_region;
""").df()

,reporting_level,reporting_company,energy_source_name,tso_region,asset_count,generation_MWh,avg_temperature_C
0,Holding,electricville AG,Biomass,50Hertz,70,553213.08,10.03
1,Holding,electricville AG,Biomass,Amprion,5,17614.69,11.62
2,Holding,electricville AG,Biomass,TenneT,54,394532.61,10.58
3,Holding,electricville AG,Biomass,TransnetBW,2,6639.62,10.20
4,Holding,electricville AG,Hydro,TenneT,3,6400.00,11.02
5,Holding,electricville AG,Solar,50Hertz,17,2381.41,10.39
6,Holding,electricville AG,Solar,Amprion,10,22406.53,10.99
7,Holding,electricville AG,Solar,TenneT,100,29350.44,10.94
8,Holding,electricville AG,Solar,TransnetBW,5,761.63,11.73
9,Holding,electricville AG,Wind,50Hertz,143,436866.87,10.31


In [12]:
con.close()
print(f"DuckDB mart created at: {DB_PATH}")

DuckDB mart created at: ../database/renewables_climate_mart.duckdb
